## CEAS-08 Dataset, Synthetic with Rewriting

In [ ]:
# SVM Classification with Synthetic Data Ratio Analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gzip
from pathlib import Path
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Configuration
PROJECT_ROOT = Path("../")  # Adjust path as needed
TRAIN_FILE = PROJECT_ROOT / "raw" / "email_phishing_CEAS-08_train.csv.gz"
TEST_FILE = PROJECT_ROOT / "raw" / "email_phishing_CEAS-08_test.csv.gz"
SYNTHETIC_FILE = PROJECT_ROOT / "data" / "seeds-augment" / "email_phishing_CEAS-08_train_malicious_rewrite_1K.csv.gz"

FIXED_TRAINING_SIZE = 2000  # Fixed total training size
SYNTHETIC_RATIOS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]  # 0% to 100%

print("Loading datasets...")

# Load original training data
with gzip.open(TRAIN_FILE, 'rt', encoding='utf-8') as f:
    train_df = pd.read_csv(f)

# Load test data
with gzip.open(TEST_FILE, 'rt', encoding='utf-8') as f:
    test_df = pd.read_csv(f)

# Load synthetic data
with gzip.open(SYNTHETIC_FILE, 'rt', encoding='utf-8') as f:
    synthetic_df = pd.read_csv(f)

print(f"Original training data: {len(train_df)} samples")
print(f"Test data: {len(test_df)} samples")
print(f"Synthetic data: {len(synthetic_df)} samples")

# Separate original data by class
original_malicious = train_df[train_df['label'] == 1].copy()
original_benign = train_df[train_df['label'] == 0].copy()

print(f"Original malicious: {len(original_malicious)}")
print(f"Original benign: {len(original_benign)}")

# Prepare test data
test_df['combined_text'] = test_df['subject'].fillna('') + ' ' + test_df['body'].fillna('')
X_test = test_df['combined_text']
y_test = test_df['label']

print(f"Test set - Malicious: {sum(y_test == 1)}, Benign: {sum(y_test == 0)}")

# Function to create training data with specific synthetic ratio
def create_training_data(synthetic_ratio, fixed_size=FIXED_TRAINING_SIZE):
    """Create training data with specified synthetic ratio"""
    
    # Calculate number of synthetic malicious samples
    n_synthetic_malicious = int(fixed_size * synthetic_ratio)
    n_original_malicious = fixed_size - n_synthetic_malicious
    
    # Sample original malicious data
    if n_original_malicious > 0:
        if len(original_malicious) >= n_original_malicious:
            sampled_original_mal = original_malicious.sample(n=n_original_malicious, random_state=42)
        else:
            sampled_original_mal = original_malicious
    else:
        sampled_original_mal = pd.DataFrame(columns=original_malicious.columns)
    
    # Sample synthetic malicious data
    if n_synthetic_malicious > 0:
        if len(synthetic_df) >= n_synthetic_malicious:
            sampled_synthetic_mal = synthetic_df.sample(n=n_synthetic_malicious, random_state=42)
        else:
            sampled_synthetic_mal = synthetic_df
    else:
        sampled_synthetic_mal = pd.DataFrame(columns=synthetic_df.columns)
    
    # Combine malicious data
    malicious_data = pd.concat([sampled_original_mal, sampled_synthetic_mal], ignore_index=True)
    
    # Sample equal number of benign data
    n_benign = len(malicious_data)
    if len(original_benign) >= n_benign:
        sampled_benign = original_benign.sample(n=n_benign, random_state=42)
    else:
        sampled_benign = original_benign
    
    # Combine all training data
    training_data = pd.concat([malicious_data, sampled_benign], ignore_index=True)
    
    print(f"Synthetic ratio {synthetic_ratio:.1f}: Total={len(training_data)}, " +
          f"Original_mal={len(sampled_original_mal)}, Synthetic_mal={len(sampled_synthetic_mal)}, " +
          f"Benign={len(sampled_benign)}")
    
    return training_data

# Function to train and evaluate SVM
def train_evaluate_svm(train_data):
    """Train SVM and evaluate on test set"""
    
    # Prepare training data
    train_data['combined_text'] = train_data['subject'].fillna('') + ' ' + train_data['body'].fillna('')
    X_train = train_data['combined_text']
    y_train = train_data['label']
    
    # Create SVM pipeline with TF-IDF
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2))),
        ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42))
    ])
    
    # Train model
    pipeline.fit(X_train, y_train)
    
    # Predict on test set
    y_pred = pipeline.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    return accuracy, precision, recall, f1

# Function to create real-world only training data with malicious ratio
def create_realworld_training_data(malicious_ratio, fixed_size=FIXED_TRAINING_SIZE):
    """Create training data using only real-world data with specified malicious ratio"""
    
    # Calculate number of malicious and benign samples
    n_malicious = int(fixed_size * malicious_ratio)
    n_benign = fixed_size - n_malicious
    
    # Sample real malicious data
    if n_malicious > 0:
        if len(original_malicious) >= n_malicious:
            sampled_malicious = original_malicious.sample(n=n_malicious, random_state=42)
        else:
            sampled_malicious = original_malicious
    else:
        sampled_malicious = pd.DataFrame(columns=original_malicious.columns)
    
    # Sample real benign data
    if n_benign > 0:
        if len(original_benign) >= n_benign:
            sampled_benign = original_benign.sample(n=n_benign, random_state=42)
        else:
            sampled_benign = original_benign
    else:
        sampled_benign = pd.DataFrame(columns=original_benign.columns)
    
    # Combine training data
    training_data = pd.concat([sampled_malicious, sampled_benign], ignore_index=True)
    
    print(f"Real-world malicious ratio {malicious_ratio:.1f}: Total={len(training_data)}, " +
          f"Malicious={len(sampled_malicious)}, Benign={len(sampled_benign)}")
    
    return training_data

# Run experiments for synthetic data ratios
print("\nRunning SVM experiments with synthetic data...")
synthetic_results = []

for ratio in SYNTHETIC_RATIOS:
    print(f"\nExperiment: Synthetic ratio = {ratio:.1f}")
    
    # Create training data
    train_data = create_training_data(ratio)
    
    # Train and evaluate
    acc, prec, rec, f1 = train_evaluate_svm(train_data)
    
    synthetic_results.append({
        'ratio': ratio,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'type': 'synthetic'
    })
    
    print(f"Results - Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

# Run experiments for real-world data only with different malicious ratios
print("\nRunning SVM experiments with real-world data only...")
realworld_results = []

for ratio in SYNTHETIC_RATIOS:
    print(f"\nExperiment: Real-world malicious ratio = {ratio:.1f}")
    
    # Create training data
    train_data = create_realworld_training_data(ratio)
    
    # Train and evaluate
    acc, prec, rec, f1 = train_evaluate_svm(train_data)
    
    realworld_results.append({
        'ratio': ratio,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'type': 'real-world'
    })
    
    print(f"Results - Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

# Convert results to DataFrames
synthetic_df = pd.DataFrame(synthetic_results)
realworld_df = pd.DataFrame(realworld_results)
all_results_df = pd.concat([synthetic_df, realworld_df], ignore_index=True)

print("\nSynthetic Data Results:")
print(synthetic_df)
print("\nReal-world Data Results:")
print(realworld_df)

# Comparison plots
plt.figure(figsize=(16, 12))

metrics = ['accuracy', 'precision', 'recall', 'f1_score']
titles = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = {'synthetic': 'blue', 'real-world': 'red'}

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, (metric, title) in enumerate(zip(metrics, titles)):
    row = i // 2
    col = i % 2
    
    # Plot synthetic data results
    synthetic_data = synthetic_df[synthetic_df['type'] == 'synthetic']
    axes[row, col].plot(synthetic_data['ratio'] * 100, synthetic_data[metric], 
                       marker='o', linewidth=2, markersize=8, color='blue', 
                       label='Synthetic Data Augmentation')
    
    # Plot real-world data results
    realworld_data = realworld_df[realworld_df['type'] == 'real-world']
    axes[row, col].plot(realworld_data['ratio'] * 100, realworld_data[metric], 
                       marker='s', linewidth=2, markersize=8, color='red', 
                       label='Real-world Data Only')
    
    axes[row, col].set_xlabel('Ratio (%)')
    axes[row, col].set_ylabel(title)
    axes[row, col].set_title(f'SVM {title} Comparison')
    axes[row, col].grid(True, alpha=0.3)
    axes[row, col].set_xlim(0, 100)
    axes[row, col].legend()
    
    # Add value labels on points
    for x, y in zip(synthetic_data['ratio'] * 100, synthetic_data[metric]):
        axes[row, col].annotate(f'{y:.3f}', (x, y), textcoords="offset points", 
                               xytext=(0,15), ha='center', fontsize=8, color='blue')
    
    for x, y in zip(realworld_data['ratio'] * 100, realworld_data[metric]):
        axes[row, col].annotate(f'{y:.3f}', (x, y), textcoords="offset points", 
                               xytext=(0,-15), ha='center', fontsize=8, color='red')

plt.tight_layout()
plt.suptitle('SVM Performance: Synthetic Data Augmentation vs Real-world Data Only', fontsize=16, y=1.02)
plt.show()

# Combined plot - all metrics
plt.figure(figsize=(16, 10))

# Subplot for each data type
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Synthetic data plot
for metric, title in zip(metrics, titles):
    ax1.plot(synthetic_df['ratio'] * 100, synthetic_df[metric], 
             marker='o', linewidth=2, markersize=6, label=title)

ax1.set_xlabel('Synthetic Data Ratio (%)')
ax1.set_ylabel('Score')
ax1.set_title('Synthetic Data Augmentation\n(Fixed training size, varying synthetic ratio)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 100)

# Real-world data plot
for metric, title in zip(metrics, titles):
    ax2.plot(realworld_df['ratio'] * 100, realworld_df[metric], 
             marker='s', linewidth=2, markersize=6, label=title)

ax2.set_xlabel('Malicious Data Ratio (%)')
ax2.set_ylabel('Score')
ax2.set_title('Real-world Data Only\n(Fixed training size, varying malicious ratio)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 100)

plt.tight_layout()
plt.show()

# Best results comparison
print("\nComparison Summary:")
print("="*60)

print("\nSynthetic Data Augmentation:")
best_synthetic = synthetic_df.loc[synthetic_df['f1_score'].idxmax()]
print(f"Best F1-Score: {best_synthetic['f1_score']:.4f} at {best_synthetic['ratio']*100:.0f}% synthetic ratio")
print(f"  Accuracy: {best_synthetic['accuracy']:.4f}")
print(f"  Precision: {best_synthetic['precision']:.4f}")
print(f"  Recall: {best_synthetic['recall']:.4f}")

print("\nReal-world Data Only:")
best_realworld = realworld_df.loc[realworld_df['f1_score'].idxmax()]
print(f"Best F1-Score: {best_realworld['f1_score']:.4f} at {best_realworld['ratio']*100:.0f}% malicious ratio")
print(f"  Accuracy: {best_realworld['accuracy']:.4f}")
print(f"  Precision: {best_realworld['precision']:.4f}")
print(f"  Recall: {best_realworld['recall']:.4f}")

# Performance difference
f1_improvement = best_synthetic['f1_score'] - best_realworld['f1_score']
print(f"\nF1-Score Improvement with Synthetic Data: {f1_improvement:+.4f}")

# Create comparison table
comparison_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Synthetic_Best': [best_synthetic['accuracy'], best_synthetic['precision'], 
                      best_synthetic['recall'], best_synthetic['f1_score']],
    'RealWorld_Best': [best_realworld['accuracy'], best_realworld['precision'], 
                      best_realworld['recall'], best_realworld['f1_score']]
})
comparison_table['Improvement'] = comparison_table['Synthetic_Best'] - comparison_table['RealWorld_Best']

print("\nDetailed Comparison:")
print(comparison_table)